In [2]:
# analyze_heatmap_patterns.py
# Pattern mining across sessions/mice/clusters for spatial firing-rate heatmaps
# Author: Laurence-ready (uses your project modules)

import os
import sys
import math
import json
import itertools
from dataclasses import dataclass, asdict, fields
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# -----------------------------
# Project imports (assumed available in your env)
# -----------------------------
try:
    from behave_analysis.process.session import get_experiment
    from behave_analysis.analyze.filtering_data.filtering_functions import filter_video_dataframe
    from behave_analysis.visualize.visualize_utils import open_tracking_data
except ImportError:
    # Optional fallback path if running from a different working directory
    sys.path.append(r"c:\Users\laurence\Documents\JAL2_subgoal_pipeline")
    from behave_analysis.process.session import get_experiment
    from behave_analysis.analyze.filtering_data.filtering_functions import filter_video_dataframe
    from behave_analysis.visualize.visualize_utils import open_tracking_data

# -----------------------------
# ---- USER CONFIG ------------
# -----------------------------

# <<<<<< Paste/maintain your session objects and groupings here >>>>>>
# Example (based on your snippet — make sure these are imported above your run):
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept
from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept
from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept
from behave_analysis.database.Experiments.JAL006_ex import JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr
from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr, JAL7_30apr
from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_tiny_3may, JAL8_flip4_10may, JAL8_14may, JAL8_21may

experiments_objects = [
    JAL6_flip7_1apr, JAL6_flip3_18mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_28mar,
    JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept,
    JAL005_8thSept, JAL005_21stSept,
    JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr, JAL7_30apr,
    JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_tiny_3may, JAL8_flip4_10may, JAL8_14may, JAL8_21may,
    JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept
]

mice_groups = {
    "JAL6": ['JAL6_flip7_1apr', 'JAL6_flip3_18mar', 'JAL6_flip4_21mar', 'JAL6_flip5_25mar', 'JAL6_28mar'],
    "JAL3": ['JAL3_25aug', 'JAL3_1sept', 'JAL3_4sept', 'JAL3_7sept'],
    "JAL7": ['JAL7_sesh8_9apr', 'JAL7_sesh9_16apr', 'JAL7_flip5_22mar', 'JAL7_flip2_12mar', 'JAL7_23apr', 'JAL7_30apr'],
    "JAL8": ['JAL8_flip1_25apr', 'JAL8_flip2_29apr', 'JAL8_tiny_3may', 'JAL8_flip4_10may', 'JAL8_14may', 'JAL8_21may'],
    "JAL4": ['JAL4_3rdSept', 'JAL4_19thSept', 'JAL4_28aug', 'JAL4_11thSept'],
    # "JAL5": ['JAL5_8thSept', 'JAL5_21stSept'],  # Include if sessions exist
}

# Conditions we’ll analyze via your filtering helper keys
CONDITION_KEYS = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]

# Binning / plotting
NBINS = 30
FPS = 40
DPI = 300
COLORMAP = "magma"

# Root save dir
SAVE_ROOT = r"Z:\Laurence\thesis\efizz_chapter\pattern_mining"
os.makedirs(SAVE_ROOT, exist_ok=True)

# -----------------------------
# ----- UTILITIES -------------
# -----------------------------

def add_bins_edges_from_positions(x: np.ndarray, y: np.ndarray, nbins=NBINS):
    x_min, x_max = np.nanmin(x), np.nanmax(x)
    y_min, y_max = np.nanmin(y), np.nanmax(y)
    eps = 1e-9
    x_edges = np.linspace(x_min, x_max + eps, nbins + 1)
    y_edges = np.linspace(y_min, y_max + eps, nbins + 1)
    return x_edges, y_edges

def apply_bins_pd(df_pd: pd.DataFrame, x_edges, y_edges, x_col="x", y_col="y"):
    nbins_x = len(x_edges) - 1
    nbins_y = len(y_edges) - 1
    df_pd["x_bins"] = np.clip(np.digitize(df_pd[x_col].values, x_edges) - 1, 0, nbins_x - 1)
    df_pd["y_bins"] = np.clip(np.digitize(df_pd[y_col].values, y_edges) - 1, 0, nbins_y - 1)

def spikes_per_sec_grid(df_cond: pd.DataFrame, df_cond_clu: pd.DataFrame, nbins=NBINS, fps=FPS, spike_col="spike_count"):
    occ = df_cond.groupby(["y_bins", "x_bins"]).size().unstack(fill_value=0)
    occ = occ.reindex(index=np.arange(nbins), columns=np.arange(nbins), fill_value=0)
    if df_cond_clu.empty:
        spk = pd.DataFrame(0, index=occ.index, columns=occ.columns)
    else:
        spk = df_cond_clu.groupby(["y_bins", "x_bins"])[spike_col].sum().unstack(fill_value=0)
        spk = spk.reindex(index=occ.index, columns=occ.columns, fill_value=0)
    with np.errstate(divide="ignore", invalid="ignore"):
        rate = spk.values / np.where(occ.values == 0, np.nan, occ.values) * fps
    return rate  # ndarray (NBINS, NBINS) with NaNs where no occupancy

def nanpearson(a, b):
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 3:
        return np.nan
    aa = a[mask] - np.nanmean(a[mask])
    bb = b[mask] - np.nanmean(b[mask])
    denom = (np.sqrt(np.nansum(aa**2)) * np.sqrt(np.nansum(bb**2)))
    return float(np.nansum(aa * bb) / denom) if denom > 0 else np.nan

def cosine_sim(a, b):
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() == 0:
        return np.nan
    aa = a[mask]; bb = b[mask]
    denom = (np.sqrt((aa**2).sum()) * np.sqrt((bb**2).sum()))
    return float((aa @ bb) / denom) if denom > 0 else np.nan

def weighted_centroid(rate_map: np.ndarray, x_edges, y_edges):
    # center coords per bin
    xs = (x_edges[:-1] + x_edges[1:]) / 2.0
    ys = (y_edges[:-1] + y_edges[1:]) / 2.0
    X, Y = np.meshgrid(xs, ys)
    R = np.nan_to_num(rate_map, nan=0.0)
    S = R.sum()
    if S <= 0:
        return np.nan, np.nan
    cx = float((R * X).sum() / S)
    cy = float((R * Y).sum() / S)
    return cx, cy

def euclid(a, b):
    ax, ay = a
    bx, by = b
    if any([not np.isfinite(ax), not np.isfinite(ay), not np.isfinite(bx), not np.isfinite(by)]):
        return np.nan
    return float(np.hypot(ax - bx, ay - by))

def robust_min_max(arrs, low=2, high=98):
    flat = np.concatenate([np.asarray(a).ravel() for a in arrs if a is not None])
    finite_vals = flat[np.isfinite(flat)]
    if finite_vals.size == 0:
        return 0.0, 1.0
    vmin = np.percentile(finite_vals, low)
    vmax = np.percentile(finite_vals, high)
    if math.isclose(vmin, vmax):
        vmax = vmin + 1e-6
    return float(vmin), float(vmax)

def safe_get_shelter_and_barrier(tracking_df: pd.DataFrame, session_obj) -> Tuple[Tuple[float,float], Optional[Tuple[float,float]]]:
    # shelter from tracking, barrier from session if available
    shelter = (np.nan, np.nan)
    barrier = (np.nan, np.nan)
    if "shelter_loc" in tracking_df.columns:
        try:
            s = tracking_df["shelter_loc"].iloc[0]
            # shelter_loc may be array-like or dict; handle common patterns
            if isinstance(s, (list, tuple, np.ndarray)) and len(s) >= 2:
                shelter = (float(s[0]), float(s[1]))
            elif isinstance(s, dict) and "x" in s and "y" in s:
                shelter = (float(s["x"]), float(s["y"]))
        except Exception:
            pass
    # barrier from session object if present
    try:
        if hasattr(session_obj, "barrier_location") and session_obj.barrier_location is not None:
            bx, by = session_obj.barrier_location[:2]
            barrier = (float(bx), float(by))
    except Exception:
        pass
    return shelter, barrier

# -----------------------------
# ----- FEATURE SCHEMA --------
# -----------------------------
@dataclass
class HeatmapFeatureRow:
    mouse: str
    session_name: str
    cluster_id: int
    condition: str
    total_spikes: int
    peak_x: float
    peak_y: float
    peak_rate: float
    centroid_x: float
    centroid_y: float
    dist_peak_to_shelter: float
    dist_centroid_to_shelter: float
    dist_peak_to_barrier: float
    dist_centroid_to_barrier: float
    frac_area_above_50pct: float
    frac_area_above_thresh: float  # absolute threshold (75th percentile of this map)
    map_sum_rate: float
    map_mean_rate: float

# -----------------------------
# ----- MAIN ANALYSIS ---------
# -----------------------------

def analyze_all():
    per_cluster_maps_dir = os.path.join(SAVE_ROOT, "per_cluster_maps")
    mouse_summaries_dir = os.path.join(SAVE_ROOT, "mouse_summaries")
    features_dir = os.path.join(SAVE_ROOT, "features")
    for d in [per_cluster_maps_dir, mouse_summaries_dir, features_dir]:
        os.makedirs(d, exist_ok=True)

    feature_rows: List[HeatmapFeatureRow] = []
    flat_records_for_similarity = []  # for within-cluster cross-condition similarity

    # ---------------- Loop sessions ----------------
    for sess in experiments_objects:
        session_obj = get_experiment(sess)
        session_name = getattr(session_obj, "name", str(session_obj))
        # Infer mouse from mapping
        mouse = next((m for m, names in mice_groups.items() if session_name in names), "UNKNOWN")

        # Paths inside the experiment
        processed_dir = getattr(session_obj, "processed_dir", None)
        if processed_dir is None:
            # Fallback: many of your sessions store processed data under 'processed_data' in the session folder
            try:
                processed_dir = os.path.join(session_obj.data_dir, "processed_data")
            except Exception:
                continue

        # Load data sources
        video_csv = os.path.join(processed_dir, "full_video_dataframe.csv")
        spikes_csv = os.path.join(processed_dir, "spike_count_by_frame_and_goodcluster.csv")

        if not (os.path.exists(video_csv) and os.path.exists(spikes_csv)):
            print(f"[WARN] Missing csv for session {session_name}")
            continue

        # Polars read
        vdf_pl = pl.read_csv(video_csv)
        sdf_pl = pl.read_csv(spikes_csv)

        # make boolean cols consistent for filtering helper
        for bcol in ["shelter","barrier_present","barrier_flipped","EscapePeriod","OutofshelterIdx","homingPeriod"]:
            if bcol in vdf_pl.columns:
                vdf_pl = vdf_pl.with_columns(pl.col(bcol).cast(pl.Boolean))

        # tracking (for shelter etc.)
        try:
            tracking_data = open_tracking_data(session_obj)
            tracking_pd = pd.DataFrame(tracking_data)
        except Exception:
            tracking_pd = pd.DataFrame()

        # Build unified position table for edges
        vpos = vdf_pl.select(["frames","mouse_x_position","mouse_y_position"]).rename({
            "frames": "frame", "mouse_x_position": "x", "mouse_y_position": "y"
        })
        vpos_pd = vpos.to_pandas()
        if vpos_pd.empty:
            print(f"[WARN] No positions for {session_name}")
            continue

        x_edges, y_edges = add_bins_edges_from_positions(vpos_pd["x"].values, vpos_pd["y"].values, nbins=NBINS)

        # Spikes DF
        sdf = sdf_pl.rename({"spike_aligned_to_frame": "frame"}).with_columns(pl.col("frame").cast(pl.Int64))
        sdf_pd = sdf.select(["frame","spike_count","spike_clusters"]).to_pandas()
        sdf_pd["spike_clusters"] = sdf_pd["spike_clusters"].astype(int)

        cluster_ids = sorted(sdf_pd["spike_clusters"].unique().astype(int))

        # shelter / barrier
        shelter_xy, barrier_xy = safe_get_shelter_and_barrier(tracking_pd, session_obj)

        # ---- per condition compute maps for all clusters ----
        for cond_key in CONDITION_KEYS:
            try:
                vdf_cond_pl = filter_video_dataframe(vdf_pl, cond_key, outofshelter=True, exclude_escape=True)
            except Exception:
                # fallback: if your helper requires different kwargs, try without flags
                vdf_cond_pl = filter_video_dataframe(vdf_pl, cond_key)

            vdf_cond = vdf_cond_pl.select(["frames","mouse_x_position","mouse_y_position"]).rename({
                "frames":"frame","mouse_x_position":"x","mouse_y_position":"y"
            }).with_columns(pl.col("frame").cast(pl.Int64))
            df_cond = vdf_cond.to_pandas()
            if df_cond.empty:
                continue

            # Merge spikes
            df_cond = df_cond.merge(sdf_pd, on="frame", how="left")
            df_cond["spike_count"] = df_cond["spike_count"].fillna(0)

            # Bin positions using global edges
            apply_bins_pd(df_cond, x_edges, y_edges)

            # Shared vmin/vmax computed later across clusters
            rate_maps_this_condition = []

            # Compute map per cluster
            for clu in cluster_ids:
                df_cond_clu = df_cond[df_cond["spike_clusters"] == clu]
                tot_spk = int(df_cond_clu["spike_count"].sum())
                rate = spikes_per_sec_grid(df_cond, df_cond_clu, nbins=NBINS, fps=FPS, spike_col="spike_count")

                # record for cross-condition similarity later
                flat_records_for_similarity.append({
                    "mouse": mouse, "session": session_name, "cluster_id": clu, "condition": cond_key,
                    "flat": rate.ravel()
                })

                # Feature extraction
                rate_nan = np.where(np.isfinite(rate), rate, np.nan)
                # peak location
                if np.isfinite(rate_nan).any():
                    idx = np.nanargmax(rate_nan)
                    peak_r = float(np.nanmax(rate_nan))
                    py, px = np.unravel_index(idx, rate_nan.shape)
                    # bin centers
                    px_coord = (x_edges[px] + x_edges[px+1]) / 2.0
                    py_coord = (y_edges[py] + y_edges[py+1]) / 2.0
                else:
                    peak_r, px_coord, py_coord = (np.nan, np.nan, np.nan)

                cx, cy = weighted_centroid(rate_nan, x_edges, y_edges)

                # distances
                d_peak_shel = euclid((px_coord, py_coord), shelter_xy)
                d_cent_shel = euclid((cx, cy), shelter_xy)
                d_peak_bar  = euclid((px_coord, py_coord), barrier_xy)
                d_cent_bar  = euclid((cx, cy), barrier_xy)

                # coverage metrics
                finite_vals = rate_nan[np.isfinite(rate_nan)]
                if finite_vals.size > 0:
                    half_peak = 0.5 * np.nanmax(finite_vals)
                    frac_half = float(np.mean(rate_nan >= half_peak))
                    thr = np.nanpercentile(finite_vals, 75.0)
                    frac_thr = float(np.mean(rate_nan >= thr))
                    msum = float(np.nansum(rate_nan))
                    mmean = float(np.nanmean(rate_nan))
                else:
                    frac_half = frac_thr = msum = mmean = np.nan

                feature_rows.append(HeatmapFeatureRow(
                    mouse=mouse, session_name=session_name, cluster_id=clu, condition=cond_key,
                    total_spikes=tot_spk,
                    peak_x=px_coord, peak_y=py_coord, peak_rate=peak_r,
                    centroid_x=cx, centroid_y=cy,
                    dist_peak_to_shelter=d_peak_shel,
                    dist_centroid_to_shelter=d_cent_shel,
                    dist_peak_to_barrier=d_peak_bar,
                    dist_centroid_to_barrier=d_cent_bar,
                    frac_area_above_50pct=frac_half,
                    frac_area_above_thresh=frac_thr,
                    map_sum_rate=msum, map_mean_rate=mmean,
                ))

            # Per-condition figure: normalize color scale across clusters for this condition
            # (optional: comment out if too heavy)
            condition_maps = [rec["flat"].reshape(NBINS, NBINS) for rec in flat_records_for_similarity
                              if rec["session"] == session_name and rec["condition"] == cond_key]
            if condition_maps:
                vmin, vmax = robust_min_max(condition_maps, 2, 98)

                # Create a quick montage of a subset (e.g., first 12 clusters)
                n_show = min(12, len(condition_maps))
                ncols = 6
                nrows = math.ceil(n_show / ncols)
                fig, axes = plt.subplots(nrows, ncols, figsize=(2.2*ncols, 2.2*nrows), sharex=True, sharey=True)
                axes = np.atleast_2d(axes)
                xs = (x_edges[:-1] + x_edges[1:]) / 2.0
                ys = (y_edges[:-1] + y_edges[1:]) / 2.0

                for i in range(n_show):
                    ax = axes[i // ncols, i % ncols]
                    im = ax.imshow(np.ma.array(condition_maps[i], mask=~np.isfinite(condition_maps[i])),
                                   origin="lower", vmin=vmin, vmax=vmax, cmap=COLORMAP,
                                   extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
                                   interpolation="nearest", aspect="equal")
                    # draw shelter/barrier markers
                    if np.isfinite(shelter_xy[0]) and np.isfinite(shelter_xy[1]):
                        ax.scatter([shelter_xy[0]], [shelter_xy[1]], s=20, marker="^", edgecolor="k", facecolor="none")
                    if np.isfinite(barrier_xy[0]) and np.isfinite(barrier_xy[1]):
                        ax.scatter([barrier_xy[0]], [barrier_xy[1]], s=20, marker="s", edgecolor="k", facecolor="none")

                    ax.set_xticks([]); ax.set_yticks([])
                for j in range(n_show, nrows*ncols):
                    axes[j // ncols, j % ncols].axis("off")

                fig.suptitle(f"{session_name} • {cond_key} (subset of cluster maps)")
                fig.tight_layout()
                outpath = os.path.join(per_cluster_maps_dir, f"{session_name}__{cond_key}__subset.eps")
                fig.savefig(outpath, format="eps", dpi=DPI, bbox_inches="tight")
                plt.close(fig)

    # ---------------- Save features table ----------------
    feature_rows_dicts = [asdict(r) for r in feature_rows]
    if feature_rows_dicts:
        features_df = pd.DataFrame(feature_rows_dicts)
    else:
        # Ensure schema exists even when no rows were generated
        features_df = pd.DataFrame(columns=[f.name for f in fields(HeatmapFeatureRow)])
    feat_csv = os.path.join(SAVE_ROOT, "features", "heatmap_features.csv")
    features_df.to_csv(feat_csv, index=False)
    print(f"[OK] Saved features: {feat_csv}")

    # ---------------- Within-mouse: cluster similarity & k-means on features ----------------
    # Compute cross-condition similarity per (mouse, session, cluster)
    sim_rows = []
    by_key = {}
    for rec in flat_records_for_similarity:
        key = (rec["mouse"], rec["session"], rec["cluster_id"])
        by_key.setdefault(key, {})[rec["condition"]] = rec["flat"]

    for (mouse, session, clu), cond_map in by_key.items():
        # pairwise similarities among available conditions
        conds = sorted(cond_map.keys())
        for (c1, c2) in itertools.combinations(conds, 2):
            a = cond_map[c1]
            b = cond_map[c2]
            sim_rows.append({
                "mouse": mouse, "session": session, "cluster_id": clu,
                "cond1": c1, "cond2": c2,
                "pearson": nanpearson(a, b),
                "cosine": cosine_sim(a, b),
            })

    sim_df = pd.DataFrame(sim_rows)
    sim_csv = os.path.join(SAVE_ROOT, "features", "cluster_level_summary.csv")
    sim_df.to_csv(sim_csv, index=False)
    print(f"[OK] Saved cluster-level condition similarity: {sim_csv}")

    # Quick per-mouse similarity visualization (mean Pearson between conditions)
    for mouse in sorted(features_df["mouse"].unique()):
        sub = sim_df[sim_df["mouse"] == mouse]
        if sub.empty:
            continue
        pivot = sub.pivot_table(index="cond1", columns="cond2", values="pearson", aggfunc="mean")
        # make symmetric
        pivot2 = pivot.copy()
        for c1 in CONDITION_KEYS:
            for c2 in CONDITION_KEYS:
                if c1 == c2:
                    pivot2.loc[c1, c2] = 1.0
                else:
                    v = np.nanmean([pivot.get(c1, {}).get(c2, np.nan), pivot.get(c2, {}).get(c1, np.nan)])
                    pivot2.loc[c1, c2] = v
        fig = plt.figure(figsize=(4.5, 4))
        ax = fig.add_subplot(111)
        im = ax.imshow(pivot2.values, origin="lower", vmin=-1, vmax=1, cmap="coolwarm")
        ax.set_xticks(range(len(pivot2.columns))); ax.set_xticklabels(pivot2.columns, rotation=30)
        ax.set_yticks(range(len(pivot2.index))); ax.set_yticklabels(pivot2.index)
        ax.set_title(f"{mouse} • mean Pearson similarity")
        fig.colorbar(im, ax=ax, shrink=0.8)
        outpath = os.path.join(SAVE_ROOT, "mouse_summaries", f"{mouse}__condition_similarity.eps")
        fig.savefig(outpath, format="eps", dpi=DPI, bbox_inches="tight")
        plt.close(fig)

    # ---------------- Across mice: pattern discovery on features ----------------
    # Build a cluster-level feature vector by aggregating across conditions
    # Example aggregations: mean/var of centroid-to-shelter, peak-to-shelter, cosine between conditions, etc.
    if not features_df.empty:
        agg_feats = features_df.groupby(["mouse","session_name","cluster_id"]).agg({
            "dist_peak_to_shelter": ["mean","std"],
            "dist_centroid_to_shelter": ["mean","std"],
            "dist_peak_to_barrier": ["mean","std"],
            "dist_centroid_to_barrier": ["mean","std"],
            "frac_area_above_50pct": ["mean","std"],
            "frac_area_above_thresh": ["mean","std"],
            "map_mean_rate": ["mean","std"],
            "map_sum_rate": ["mean","std"],
            "peak_rate": ["mean","std"],
        })
        agg_feats.columns = ["__".join(col) for col in agg_feats.columns]
        agg_feats = agg_feats.reset_index()

        # Merge an average of between-condition similarity per cluster
        if not sim_df.empty:
            sim_agg = sim_df.groupby(["mouse","session","cluster_id"]).agg({
                "pearson":"mean", "cosine":"mean"
            }).reset_index().rename(columns={"session":"session_name",
                                             "pearson":"mean_cond_pearson",
                                             "cosine":"mean_cond_cosine"})
            agg_feats = agg_feats.merge(sim_agg, on=["mouse","session_name","cluster_id"], how="left")
        else:
            agg_feats["mean_cond_pearson"] = np.nan
            agg_feats["mean_cond_cosine"] = np.nan

        # PCA + kmeans on standardized features
        feat_cols = [c for c in agg_feats.columns if c not in ["mouse","session_name","cluster_id"]]
        X = agg_feats[feat_cols].values
        # handle NaNs by column imputation with nanmedian
        col_med = np.nanmedian(X, axis=0)
        inds = np.where(~np.isfinite(X))
        X[inds] = np.take(col_med, inds[1])

        scaler = StandardScaler()
        Xs = scaler.fit_transform(X)

        n_components = min(10, Xs.shape[1])
        pca = PCA(n_components=n_components, random_state=0)
        Z = pca.fit_transform(Xs)

        k = min(6, max(2, int(np.sqrt(len(agg_feats) / 4))))  # heuristic
        km = KMeans(n_clusters=k, n_init="auto", random_state=0)
        labels = km.fit_predict(Z)

        agg_feats["pattern_cluster_k"] = labels
        agg_csv = os.path.join(SAVE_ROOT, "features", "across_cluster_aggregates.csv")
        agg_feats.to_csv(agg_csv, index=False)
        print(f"[OK] Saved across-cluster aggregates: {agg_csv}")

        # 2D viz (PCA 1–2)
        fig = plt.figure(figsize=(6,5))
        ax = fig.add_subplot(111)
        sc = ax.scatter(Z[:,0], Z[:,1], c=labels, s=18, alpha=0.9)
        ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
        ax.set_title("Across mice: cluster pattern map (PCA + k-means)")
        # simple legend-ish
        handles = []
        for lab in sorted(np.unique(labels)):
            handles.append(plt.Line2D([], [], marker='o', linestyle='None', label=f"Pat {lab}", markersize=6))
        ax.legend(handles=handles, loc="best", frameon=False)
        outpath = os.path.join(SAVE_ROOT, "across_mice_pattern_map.png")
        fig.savefig(outpath, dpi=200, bbox_inches="tight")
        plt.close(fig)

    print("[DONE] Pattern mining complete.")

if __name__ == "__main__":
    analyze_all()

[OK] Saved features: Z:\Laurence\thesis\efizz_chapter\pattern_mining\features\heatmap_features.csv
[OK] Saved cluster-level condition similarity: Z:\Laurence\thesis\efizz_chapter\pattern_mining\features\cluster_level_summary.csv
[DONE] Pattern mining complete.
